# Improving the LLPS project model

This notebook contains a complete, reproducible baseline pipeline for your uploaded dataset. Read `README.md` for the diagnosis and limitations.

**Actual completed results:** fixed XGBoost + engineered features, held-out whole records: **71.45% accuracy / 0.789 ROC-AUC**. Nested record evaluation: **70.95% / 0.787**. Nested protein-held-out evaluation: **60.12% / 0.630**. Record-weighted accuracy on unseen proteins was only **59.42%**, below its **62.72%** majority baseline. These are different tasks; the 57% training code was absent, so this is not a controlled comparison to that run.

The CSV omits Cluster_ID. Protein-ID and exact-sequence exclusion is weaker than the homology-cluster split planned in your report. Restore clusters before making an unseen-family claim. Every score above is out-of-fold; final saved models are fitted on all data and have no separate test score.

The notebook writes four self-contained Python modules so it can run independently in Colab. Training may take several minutes on CPU. The original dataset is required; it is not included in this package.

In [ ]:
%pip install -q numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 xgboost==3.4.1 joblib==1.5.3

## 1. Install the reusable training and inference modules

In [ ]:
%%writefile llps_benchmark.py
"""Condition-aware LLPS benchmark. Does not treat sampled conditions as independent experiments."""
import os
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
import argparse, json, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, brier_score_loss, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import joblib

ENV = ['temperature_C','protein_conc_uM','pH','PEG_300_1k_pct','PEG_3k_6k_pct','PEG_8k_20k_pct','Ficoll_pct','Dextran_le40k_pct','Dextran_ge70k_pct','MgCl2_mM','NaCl_mM','KCl_mM','glycerol_pct']
MAXIMA = np.array([60,1000,14,50,50,50,50,50,50,50,2000,1000,10])

def groups_for(df):
    # Connected components ensure BOTH repeated protein IDs and identical sequences stay together.
    parent = list(range(len(df)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    for col in ['Protein_ID','Sequence','PSID'] + (['Cluster_ID'] if 'Cluster_ID' in df else []):
        seen={}
        for i,v in enumerate(df[col]):
            if pd.isna(v): raise ValueError(f'Missing {col}; repair grouping first')
            if v in seen: parent[find(i)]=find(seen[v])
            else: seen[v]=i
    return np.array([find(i) for i in range(len(df))])

def features(df, engineered=False):
    # Explicit whitelist: never learn from IDs, labels or accidental future columns.
    seq=['Sequence_length','Molecular_weight','pI','Mean_KD_hydrophobicity','Shannon_entropy','Positive_charge_count','Negative_charge_count','Net_charge','Positive_charge_frac','Negative_charge_frac','Net_charge_per_residue']
    seq += ['AA_'+a+'_frac' for a in 'ACDEFGHIKLMNPQRSTVWY']
    seq += ['HB_frac','Polar_frac','Arom_frac','Cation_frac','Anion_frac','IUPred3_mean','IUPred3_median','IUPred3_max','IUPred3_disordered_frac','LCR_length','LCR_frac','LCR_count']
    x=df[ENV+seq].astype(float).copy()
    if engineered:
        raw=df[ENV]*MAXIMA
        x['pH_minus_pI']=raw['pH']-df.pI
        x['abs_pH_minus_pI']=x.pH_minus_pI.abs()
        for c in ['protein_conc_uM','NaCl_mM','KCl_mM','MgCl2_mM']:
            x['log1p_'+c]=np.log1p(raw[c].clip(lower=0))
        x['ionic_strength_proxy_mM']=raw.NaCl_mM+raw.KCl_mM+3*raw.MgCl2_mM
        # Proxy omits buffer ions and other additives; not measured ionic strength.
        x['total_PEG_pct']=raw[['PEG_300_1k_pct','PEG_3k_6k_pct','PEG_8k_20k_pct']].sum(axis=1)
        x['KR_frac']=df.AA_K_frac+df.AA_R_frac
        # Lightweight order-dependent descriptors, computed per unique sequence.
        records=[]
        for s in df.Sequence.unique():
            q=np.array([1 if a in 'KR' else -1 if a in 'DE' else 0 for a in s],float)
            arom=np.array([a in 'FYW' for a in s],float)
            distances=np.diff(np.flatnonzero(arom))
            window=min(15,len(s))
            records.append({'Sequence':s,'charge_adjacent_product':np.mean(q[:-1]*q[1:]),
                'max_window_abs_charge':np.max(np.abs(np.convolve(q,np.ones(window)/window,'valid'))),
                'max_window_aromatic_frac':np.max(np.convolve(arom,np.ones(window)/window,'valid')),
                'aromatic_gap_mean':distances.mean() if len(distances) else len(s),
                'aromatic_gap_std':distances.std() if len(distances) else 0,
                'RG_pair_density':sum(s[i:i+2]=='RG' for i in range(len(s)-1))/len(s)})
        f=pd.DataFrame(records).set_index('Sequence')
        for c in f: x[c]=df.Sequence.map(f[c])
    return x.replace([np.inf,-np.inf],np.nan)

def weights(df):
    # PSID is the available proxy for original record; sum of weights per PSID = 1.
    return (1/df.groupby('PSID').PSID.transform('size')).to_numpy()

def score(y,p,w=None):
    z=(p>=.5).astype(int)
    return dict(accuracy=accuracy_score(y,z,sample_weight=w),balanced_accuracy=balanced_accuracy_score(y,z,sample_weight=w),roc_auc=roc_auc_score(y,p,sample_weight=w),average_precision=average_precision_score(y,p,sample_weight=w),f1=f1_score(y,z,sample_weight=w,zero_division=0),mcc=matthews_corrcoef(y,z,sample_weight=w),brier=brier_score_loss(y,p,sample_weight=w))

def model_for(name):
    if name=='logistic':return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(C=1,max_iter=2000))
    if name=='extra_trees':return make_pipeline(SimpleImputer(strategy='median'),ExtraTreesClassifier(n_estimators=250,max_features=.8,min_samples_leaf=3,n_jobs=2,random_state=42))
    if name.startswith('xgb'):
        from xgboost import XGBClassifier
        depth=int(name[-1]);return XGBClassifier(n_estimators=400,max_depth=depth,learning_rate=.04,min_child_weight=8,subsample=.85,colsample_bytree=.85,reg_lambda=10,tree_method='hist',n_jobs=2,random_state=42,eval_metric='logloss')
    leaf={'hist_small':7,'hist_medium':15,'hist_large':31}[name]
    return HistGradientBoostingClassifier(max_iter=250,max_leaf_nodes=leaf,learning_rate=.06,l2_regularization=10,min_samples_leaf=20,early_stopping=False,random_state=42)

def fit(model,x,y,w):
    # Rescale weights to mean 1 so regularization has a stable scale.
    w=w/w.mean()
    if hasattr(model,'steps'):
        model.fit(x,y,**{model.steps[-1][0]+'__sample_weight':w})
    else:model.fit(x,y,sample_weight=w)
    return model

def run(data,out,include_xgb=False):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(data).reset_index(drop=True); y=df.Label.to_numpy();g=groups_for(df)
    xs={False:features(df),True:features(df,True)}
    w=weights(df)
    keys=['Sequence']+ENV
    a=df.groupby(keys,dropna=False).Label.agg(['size','nunique','sum'])
    audit=dict(rows=len(df),PSIDs=df.PSID.nunique(),proteins=df.Protein_ID.nunique(),sequences=df.Sequence.nunique(),connected_groups=len(np.unique(g)),has_cluster_ID='Cluster_ID' in df,positive_fraction=float(y.mean()),duplicate_input_rows=len(df)-len(a),conflicting_input_groups=int((a['nunique']>1).sum()),rows_in_conflicting_groups=int(a.loc[a['nunique']>1,'size'].sum()),dataset_sha256=hashlib.sha256(Path(data).read_bytes()).hexdigest(),python=platform.python_version(),sklearn=sklearn.__version__)
    (out/'audit.json').write_text(json.dumps(audit,indent=2));print('AUDIT',audit,flush=True)
    # Nested CV: candidates selected ONLY within outer training data. Each outer fold evaluated once.
    candidates=[('logistic',False),('extra_trees',False),('hist_small',False),('hist_medium',True)]
    if include_xgb:candidates += [('xgb3',False),('xgb5',True)]
    outer=StratifiedGroupKFold(5,shuffle=True,random_state=42)
    pred=np.zeros(len(df));base=np.zeros(len(df));folds=np.zeros(len(df),int); results=[];selection=[]
    for f,(tr,te) in enumerate(outer.split(df,y,g)):
        assert not set(g[tr])&set(g[te])
        for col in ['PSID','Sequence','Protein_ID']+(['Cluster_ID'] if 'Cluster_ID' in df else []):assert not set(df.iloc[tr][col])&set(df.iloc[te][col])
        inner=list(StratifiedGroupKFold(3,shuffle=True,random_state=100+f).split(df.iloc[tr],y[tr],g[tr]))
        scores=[]
        for name,eng in candidates:
            pp=np.zeros(len(tr))
            for it,iv in inner:
                m=fit(model_for(name),xs[eng].iloc[tr[it]],y[tr[it]],weights(df.iloc[tr[it]]))
                pp[iv]=m.predict_proba(xs[eng].iloc[tr[iv]])[:,1]
            metrics=score(y[tr],pp,weights(df.iloc[tr]));scores.append(metrics['roc_auc'])
            selection.append(dict(outer_fold=f,model=name,engineered=eng,**metrics))
            print('INNER',f,name,eng,round(metrics['roc_auc'],4),flush=True)
        name,eng=candidates[int(np.argmax(scores))]
        m=fit(model_for(name),xs[eng].iloc[tr],y[tr],weights(df.iloc[tr]));pred[te]=m.predict_proba(xs[eng].iloc[te])[:,1]
        # Majority baseline is estimated only from training experiments.
        base[te]=np.average(y[tr],weights=weights(df.iloc[tr]));folds[te]=f
        results.append(dict(fold=f,selected=name,engineered=eng,train_rows=len(tr),test_rows=len(te),test_groups=len(np.unique(g[te])),test_positive_fraction=float(y[te].mean()),**score(y[te],pred[te],weights(df.iloc[te]))))
        print('OUTER',results[-1],flush=True)
    pd.DataFrame(results).to_csv(out/'outer_fold_metrics.csv',index=False)
    pd.DataFrame(selection).to_csv(out/'inner_selection.csv',index=False)
    summary={'nested_protein_grouped_row_metrics':score(y,pred),'nested_protein_grouped_PSID_weighted_metrics':score(y,pred,w),'training_majority_baseline_row_metrics':score(y,base),'training_majority_baseline_PSID_weighted_metrics':score(y,base,w),'confusion_matrix_rows':confusion_matrix(y,pred>=.5).tolist()}
    # Fixed-config row split: intentionally optimistic diagnostic, not a valid novel-protein benchmark.
    rp=np.zeros(len(df))
    for tr,te in StratifiedKFold(5,shuffle=True,random_state=42).split(df,y):
        m=fit(model_for('hist_small'),xs[False].iloc[tr],y[tr],weights(df.iloc[tr]));rp[te]=m.predict_proba(xs[False].iloc[te])[:,1]
    summary['random_row_diagnostic_ONLY']=score(y,rp)
    # Select deployment settings from inner validation results, never outer test scores.
    sel=pd.DataFrame(selection).groupby(['model','engineered']).roc_auc.mean().idxmax()
    final=fit(model_for(sel[0]),xs[sel[1]],y,w)
    joblib.dump(dict(model=final,feature_columns=xs[sel[1]].columns.tolist(),engineered=bool(sel[1]),grouping='connected Protein_ID + Sequence + PSID (+ Cluster_ID if available)',threshold=.5),out/'model.joblib')
    summary['deployment_configuration']={'model':sel[0],'engineered':bool(sel[1]),'trained_on_all_rows':True,'evaluation_note':'Metrics evaluate nested model-selection procedure; deployment model has no separate held-out score.'}
    (out/'summary.json').write_text(json.dumps(summary,indent=2))
    df[['PSID','Protein_ID','Label']].assign(group=g,fold=folds,probability=pred,random_row_probability=rp).to_csv(out/'out_of_fold_predictions.csv',index=False)
    print('SUMMARY',json.dumps(summary,indent=2),flush=True)
    return summary

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--data',required=True);p.add_argument('--out',default='results');p.add_argument('--include-xgb',action='store_true');a=p.parse_args();run(a.data,a.out,a.include_xgb)


Overwriting llps_benchmark.py


In [ ]:
%%writefile known_protein_benchmark.py
"""Secondary benchmark: held-out PSIDs with repeated proteins allowed.
Overlapping identical condition inputs are connected BEFORE splitting.
This is NOT a novel-protein or independent-paper evaluation.
"""
from llps_benchmark import *

def record_groups(df):
    p=list(range(len(df)))
    def find(i):
        while p[i]!=i:p[i]=p[p[i]];i=p[i]
        return i
    for values in [df.PSID, pd.util.hash_pandas_object(df[['Sequence']+ENV],index=False)]:
        seen={}
        for i,v in enumerate(values):
            if v in seen:p[find(i)]=find(seen[v])
            else:seen[v]=i
    return np.array([find(i) for i in range(len(df))])

def run_known(data,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    d=pd.read_csv(data);y=d.Label.to_numpy();g=record_groups(d);w=weights(d)
    allpred={};rows=[];seen=np.zeros(len(d),bool);fold=np.zeros(len(d),int)
    for name,eng in [('hist_small',False),('xgb3',False),('xgb5',True)]:
        x=features(d,eng);pr=np.zeros(len(d));baseline=np.zeros(len(d))
        for f,(tr,te) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(d,y,g)):
            assert not set(g[tr])&set(g[te]);assert not set(d.iloc[tr].PSID)&set(d.iloc[te].PSID)
            seen[te]=d.iloc[te].Sequence.isin(d.iloc[tr].Sequence).to_numpy();fold[te]=f
            m=fit(model_for(name),x.iloc[tr],y[tr],weights(d.iloc[tr]));pr[te]=m.predict_proba(x.iloc[te])[:,1]
            baseline[te]=np.average(y[tr],weights=weights(d.iloc[tr]))
        allpred[name]=pr
        for subset,ix in [('all_rows',np.ones(len(d),bool)),('sequence_seen_in_training',seen)]:
            for weighting,ww in [('row',None),('PSID',w[ix])]:
                rows.append(dict(model=name,subset=subset,weighting=weighting,n_rows=int(ix.sum()),**score(y[ix],pr[ix],ww)))
        print(name,rows[-4:],flush=True)
    rows.append(dict(model='training_majority',subset='all_rows',weighting='row',n_rows=len(d),**score(y,baseline)))
    pd.DataFrame(rows).to_csv(out/'metrics.csv',index=False)
    d[['PSID','Protein_ID','Label']].assign(fold=fold,group=g,sequence_seen_in_training=seen,**allpred).to_csv(out/'predictions.csv',index=False)
    # Predeclared fixed baseline, not selected by outer test performance.
    x=features(d);m=fit(model_for('xgb3'),x,y,w)
    joblib.dump(dict(model=m,feature_columns=x.columns.tolist(),engineered=False,threshold=.5),out/'known_protein_xgb3.joblib')
    m.save_model(out/'known_protein_xgb3.json')

if __name__=='__main__':
    import sys
    run_known(sys.argv[1],sys.argv[2])


Overwriting known_protein_benchmark.py


In [ ]:
%%writefile nested_record_benchmark.py
"""Nested model selection for held-out records; repeated proteins may occur across folds."""
from known_protein_benchmark import *

def run_nested_records(data,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    d=pd.read_csv(data);y=d.Label.to_numpy();g=record_groups(d);w=weights(d)
    xs={False:features(d),True:features(d,True)}
    candidates=[('hist_medium',True),('xgb3',False),('xgb5',True)]
    pp=np.zeros(len(d));fold=np.zeros(len(d),int);selection=[];foldrows=[];seen=np.zeros(len(d),bool)
    for f,(tr,te) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(d,y,g)):
        sc=[]
        for name,eng in candidates:
            ip=np.zeros(len(tr))
            for it,iv in StratifiedGroupKFold(3,shuffle=True,random_state=100+f).split(d.iloc[tr],y[tr],g[tr]):
                m=fit(model_for(name),xs[eng].iloc[tr[it]],y[tr[it]],weights(d.iloc[tr[it]]));ip[iv]=m.predict_proba(xs[eng].iloc[tr[iv]])[:,1]
            auc=roc_auc_score(y[tr],ip,sample_weight=weights(d.iloc[tr]));sc.append(auc)
            selection.append(dict(fold=f,model=name,engineered=eng,inner_auc=auc))
            print('INNER',f,name,auc,flush=True)
        name,eng=candidates[int(np.argmax(sc))]
        m=fit(model_for(name),xs[eng].iloc[tr],y[tr],weights(d.iloc[tr]));pp[te]=m.predict_proba(xs[eng].iloc[te])[:,1];fold[te]=f
        seen[te]=d.iloc[te].Sequence.isin(d.iloc[tr].Sequence).to_numpy()
        foldrows.append(dict(fold=f,selected=name,engineered=eng,**score(y[te],pp[te],weights(d.iloc[te]))));print('OUTER',foldrows[-1],flush=True)
    summary=dict(row_metrics=score(y,pp),PSID_weighted_metrics=score(y,pp,w),seen_sequence_row_metrics=score(y[seen],pp[seen]),seen_sequence_PSID_metrics=score(y[seen],pp[seen],w[seen]),seen_sequence_rows=int(seen.sum()),note='Nested CV; PSID and identical encoded inputs disjoint, related/repeated proteins and publications may overlap.')
    selection=pd.DataFrame(selection);name,eng=selection.groupby(['model','engineered']).inner_auc.mean().idxmax()
    m=fit(model_for(name),xs[eng],y,w)
    joblib.dump(dict(model=m,feature_columns=xs[eng].columns.tolist(),engineered=bool(eng),threshold=.5),out/'model.joblib')
    if name.startswith('xgb'):m.save_model(out/'model.json')
    summary['deployment_configuration']=dict(model=name,engineered=bool(eng),note='Trained on all data; nested metrics evaluate the selection procedure, not an independent test of this fitted model.')
    (out/'summary.json').write_text(json.dumps(summary,indent=2));selection.to_csv(out/'inner_selection.csv',index=False);pd.DataFrame(foldrows).to_csv(out/'fold_metrics.csv',index=False)
    d[['PSID','Protein_ID','Label']].assign(group=g,fold=fold,probability=pp,sequence_seen_in_training=seen).to_csv(out/'predictions.csv',index=False)
    print('SUMMARY',summary,flush=True)

if __name__=='__main__':
    import sys
    run_nested_records(sys.argv[1],sys.argv[2])


Overwriting nested_record_benchmark.py


In [ ]:
%%writefile predict.py
"""Score a CSV encoded with the SAME 56-feature schema and environmental normalization."""
import argparse
import joblib
import pandas as pd
from llps_benchmark import features

def predict(model_path,data_path,output_path):
    bundle=joblib.load(model_path)
    d=pd.read_csv(data_path)
    x=features(d,bundle['engineered'])[bundle['feature_columns']]
    p=bundle['model'].predict_proba(x)[:,1]
    keep=[c for c in ['PSID','Protein_ID'] if c in d]
    out=d[keep].copy();out['llps_model_score']=p;out['prediction_at_0_5']=(p>=bundle['threshold']).astype(int)
    out.to_csv(output_path,index=False)
    return out

if __name__=='__main__':
    a=argparse.ArgumentParser();a.add_argument('--model',required=True);a.add_argument('--data',required=True);a.add_argument('--out',required=True);x=a.parse_args();predict(x.model,x.data,x.out)


Overwriting predict.py


## 2. Select the dataset

Use `LLPSDB_v2_model_dataset.csv`, or preferably rebuild the CSV with Cluster_ID using the patch below in your original preprocessing notebook. Do not change the column normalization conventions. The environment maxima were: temperature 60, protein concentration 1000, pH 14, each crowder 50, MgCl2 50, NaCl 2000, KCl 1000, glycerol 10.

```python
cluster_map = unique_sequences[['Sequence', 'Cluster_ID']].drop_duplicates()
model_dataset = llpsense_normalized.drop(columns=['Cluster_ID'], errors='ignore').merge(
    cluster_map, on='Sequence', how='left', validate='many_to_one')
model_dataset = model_dataset.merge(
    sequence_features.drop(columns=['Sequence_ID'], errors='ignore'),
    on='Sequence', how='left', validate='many_to_one')
assert model_dataset.Cluster_ID.notna().all()
model_dataset.to_csv('LLPSDB_v2_model_dataset_with_clusters.csv', index=False)
```

Run the patch while the original notebook variables still exist. This avoids attempting to infer clusters from protein IDs.

In [ ]:
from pathlib import Path
import pandas as pd
DATA = Path('LLPSDB_v2_model_dataset.csv')  # Change to the rebuilt CSV if available.
if not DATA.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        choices = [Path(n) for n in uploaded if n.endswith('.csv')]
        assert len(choices) == 1, 'Upload exactly the model dataset CSV.'
        DATA = choices[0]
    except ImportError:
        raise FileNotFoundError('Place your dataset CSV beside this notebook or set DATA to its path.')
df = pd.read_csv(DATA)
print(df.shape, 'Unique PSIDs:', df.PSID.nunique(), 'Unique sequences:', df.Sequence.nunique())
print('Cluster_ID available:', 'Cluster_ID' in df)
print('Always-LLPS accuracy by rows:', df.Label.mean())
print('Always-LLPS accuracy by PSID:', df.drop_duplicates('PSID').Label.mean())

(20719, 60) Unique PSIDs: 2626 Unique sequences: 815
Cluster_ID available: False
Always-LLPS accuracy by rows: 0.5627684733819199
Always-LLPS accuracy by PSID: 0.6271896420411271


## 3. Main test: unseen proteins / clusters

Nested five-fold outer / three-fold inner evaluation. The connected grouping excludes shared Protein_ID, Sequence and PSID. Cluster_ID is incorporated when available. All model selection uses inner folds. Inverse-PSID weighting limits domination by synthetic range expansion. No default row-wise early-stopping split is used.

The class labels are inherited range labels, so even grouped validation is limited by the source data quality.

In [ ]:
%pip install -q numpy<2
# Please restart the Colab session (Runtime -> Restart session) after running this line to ensure the correct NumPy version is loaded, then execute the cell again.

from llps_benchmark import run
protein_summary = run(DATA, 'new_protein_results', include_xgb=True)
pd.DataFrame({
    'row': protein_summary['nested_protein_grouped_row_metrics'],
    'equal_weight_per_PSID': protein_summary['nested_protein_grouped_PSID_weighted_metrics']
})

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
contourpy 1.4.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.54.1 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which i

ModuleNotFoundError: No module named 'numpy.strings'

## 4. Secondary test: held-out experiments with proteins allowed to repeat

PSIDs and exact encoded sequence/condition inputs stay together, preventing direct duplicate-input leakage. Protein families and papers may overlap: this answers a different question from section 3. An exact-sequence-seen subset is also reported. This is still not a publication-held-out evaluation.

In [ ]:
from nested_record_benchmark import run_nested_records
run_nested_records(DATA, 'new_record_results')
import json
record_summary = json.loads(Path('new_record_results/summary.json').read_text())
pd.DataFrame({'row': record_summary['row_metrics'], 'equal_weight_per_PSID': record_summary['PSID_weighted_metrics']})

## 5. Optional: inspect the fixed XGBoost baseline

This reproduces the exploratory fixed-configuration comparison (including the depth-5 engineered model). Do not keep trying configurations against these same outer folds and then present the best result as an untouched test score.

In [ ]:
RUN_FIXED_COMPARISON = False
if RUN_FIXED_COMPARISON:
    from known_protein_benchmark import run_known
    run_known(DATA, 'fixed_record_comparison')
    display(pd.read_csv('fixed_record_comparison/metrics.csv'))

## 6. Make predictions on consistently encoded rows

The input must already contain the sequence descriptors and normalized environmental columns. This is not a raw-sequence featurization tool. Use model scores cautiously until probability calibration is checked in grouped validation.

In [ ]:
# Example after training; edit the prediction CSV path before enabling.
# from predict import predict
# result = predict('new_record_results/model.joblib',
#                  'prepared_new_conditions.csv', 'predictions.csv')
# display(result.head())

## Next experiments

1. Restore clustering and source-record IDs, and repair confirmed parsing/label problems with a correction log.
2. Extract frozen ProtT5/ESM embeddings once per unique sequence, then join to the condition rows. This extension is not implemented here.
3. Compare conditions-only, descriptors-only, combined descriptors, and embeddings plus conditions on the same grouped splits. Fit PCA/scalers only within training folds.
4. Add calibration, source/publication-held-out tests, and interpretable condition-response plots before selecting simulation candidates.

The fixed depth-5 XGBoost model can be loaded from the downloaded package under `known_protein_results/xgb5_engineered.joblib`. The command-line runner in this notebook's optional comparison saves a depth-3 XGBoost baseline, while nested runners save their independently selected configuration. All saved final models were refit on all rows.

Sources: [scikit-learn grouped CV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html), [XGBoost estimator documentation](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html), [LLPSense](https://github.com/NearNiah/LLPSense).